# Day 11 – Multi-Turn Chat API

## Objectives

- Build a `POST /chat` style chat flow
- Load conversation history
- Perform document retrieval
- Generate answers using an LLM
- Persist every conversation turn
- Handle follow-up questions
- Convert follow-up questions into context-aware queries
- Understand response streaming

## 1. Multi-Turn Conversation

### Definition

A multi-turn conversation is a conversation where the user and AI
exchange multiple messages while maintaining context from previous turns.

### Example

User:
What is the SmartHome Hub?

AI:
The SmartHome Hub is a central device for controlling smart-home devices.

User:
What are its key features?

AI:
Its key features include universal compatibility, an AI-powered assistant,
and enhanced security.

The second question depends on the first question.

In [1]:
conversation_history = [
    {
        "role": "user",
        "content": "What is the SmartHome Hub?"
    },
    {
        "role": "assistant",
        "content": (
            "The SmartHome Hub is a central device designed "
            "to connect and control smart-home devices."
        )
    }
]

conversation_history

[{'role': 'user', 'content': 'What is the SmartHome Hub?'},
 {'role': 'assistant',
  'content': 'The SmartHome Hub is a central device designed to connect and control smart-home devices.'}]

## 2. Chat History

### Definition

Chat history is the collection of previous user and assistant messages
belonging to the same conversation.

It allows the AI to understand references such as:

- "it"
- "its"
- "that"
- "the previous one"
- "what about its features?"

### Example

Previous:

User:
What is the SmartHome Hub?

AI:
It is a central smart-home device.

Current:

User:
What are its features?

The word "its" refers to the SmartHome Hub.

In [2]:
current_question = "What are its key features?"

print(current_question)

What are its key features?


## 3. Context-Aware Query

### Definition

A context-aware query is a question rewritten using information
from previous conversation turns.

Original question:

"What are its key features?"

Context-aware question:

"What are the key features of the SmartHome Hub?"

This improves document retrieval because the important subject
is explicitly included in the query.

In [3]:
context_aware_query = (
    "What are the key features of the SmartHome Hub?"
)

print(context_aware_query)

What are the key features of the SmartHome Hub?


## 4. Retrieval

### Definition

Retrieval means searching the vector database for document chunks
that are relevant to the user's question.

Our flow is:

Question
   ↓
Embedding
   ↓
Vector Search
   ↓
Relevant Chunks

In [13]:
import chromadb

client = chromadb.PersistentClient(
    path="../ingestion/chroma_db"
)

collection = client.get_or_create_collection(
    name="knowledge_base"
)

print("Total chunks:", collection.count())

Total chunks: 147


## 6. History + Retrieval + Generation

A conversational RAG system combines three important pieces:

1. Conversation history
2. Retrieved document context
3. Current question

Flow:

Conversation History
        +
Retrieved Context
        +
Current Question
        ↓
       LLM
        ↓
    Final Answer

In [17]:
results = collection.query(
    query_texts=[context_aware_query],
    n_results=5
)


In [19]:
print(context_aware_query)

What are the key features of the SmartHome Hub?


In [20]:
results = collection.query(
    query_texts=[context_aware_query],
    n_results=5
)

print(results)

{'ids': [['146', '142', '143', '144', '145']], 'embeddings': None, 'documents': [['5\nConclusion\nThe SmartHome Hub represents a significant\nopportunity for Innovative Tech Solutions to\ncapture a substantial share of the rapidly\ngrowing smart home market. With our\nunique features and strategic marketing\nplan, we are well-positioned for a successful\nproduct launch. \nNext Steps\n1. Finalize production agreements with\nmanufacturers.\n2. Launch pre-order website and marketing\ncampaign.\n3. Prepare for the official launch event.', 'Innovative Tech\nSolutions, Inc.\nProduct Launch Report:\nSmartHome Hub\nPrepared by:\nAlex Johnson\nsample-files.com\n+1-555-123-4567', 'Introduction\nThis report outlines the launch strategy for\nour new SmartHome Hub, a central device\ndesigned to connect and control all smart\nhome devices seamlessly. Our goal is to\nrevolutionize home automation and\nestablish ourselves as market leaders in this\ngrowing sector.\n2\nObjectives\n1. To introduce the S

In [21]:
retrieved_documents = results["documents"][0]

In [22]:
print(retrieved_documents)

['5\nConclusion\nThe SmartHome Hub represents a significant\nopportunity for Innovative Tech Solutions to\ncapture a substantial share of the rapidly\ngrowing smart home market. With our\nunique features and strategic marketing\nplan, we are well-positioned for a successful\nproduct launch. \nNext Steps\n1. Finalize production agreements with\nmanufacturers.\n2. Launch pre-order website and marketing\ncampaign.\n3. Prepare for the official launch event.', 'Innovative Tech\nSolutions, Inc.\nProduct Launch Report:\nSmartHome Hub\nPrepared by:\nAlex Johnson\nsample-files.com\n+1-555-123-4567', 'Introduction\nThis report outlines the launch strategy for\nour new SmartHome Hub, a central device\ndesigned to connect and control all smart\nhome devices seamlessly. Our goal is to\nrevolutionize home automation and\nestablish ourselves as market leaders in this\ngrowing sector.\n2\nObjectives\n1. To introduce the SmartHome Hub and its\nkey features.\n2. To present market research findings and\n

In [23]:
context_text = "\n\n".join(retrieved_documents)

In [24]:
history_text = ""

for message in conversation_history:
    history_text += (
        f"{message['role'].capitalize()}: "
        f"{message['content']}\n"
    )

context_text = "\n\n".join(retrieved_documents)

prompt = f"""
You are a helpful document assistant.

Conversation history:
{history_text}

Retrieved document context:
{context_text}

Current question:
{current_question}

Answer the question using the retrieved document context.
If the answer is not available in the context, say that you
do not have enough information.
"""

print(prompt)


You are a helpful document assistant.

Conversation history:
User: What is the SmartHome Hub?
Assistant: The SmartHome Hub is a central device designed to connect and control smart-home devices.


Retrieved document context:
5
Conclusion
The SmartHome Hub represents a significant
opportunity for Innovative Tech Solutions to
capture a substantial share of the rapidly
growing smart home market. With our
unique features and strategic marketing
plan, we are well-positioned for a successful
product launch. 
Next Steps
1. Finalize production agreements with
manufacturers.
2. Launch pre-order website and marketing
campaign.
3. Prepare for the official launch event.

Innovative Tech
Solutions, Inc.
Product Launch Report:
SmartHome Hub
Prepared by:
Alex Johnson
sample-files.com
+1-555-123-4567

Introduction
This report outlines the launch strategy for
our new SmartHome Hub, a central device
designed to connect and control all smart
home devices seamlessly. Our goal is to
revolutionize home aut

## 7. Generation

### Definition

Generation is the process where the LLM receives the prompt
and produces the final answer.

Our flow is:

Prompt
  ↓
LLM
  ↓
Answer

In [32]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../../.env")

api_key = os.getenv("OPENAI_API_KEY")

client_llm = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

In [33]:
response = client_llm.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Say only: API connection successful."
        }
    ]
)

print(response.choices[0].message.content)

API connection successful.


## 8. Persist Every Turn

### Definition

Persistence means saving the conversation to PostgreSQL.

For every interaction we save:

User message
+
Assistant response

Example:

User:
What are its key features?

Assistant:
The SmartHome Hub has universal compatibility,
an AI-powered assistant, and enhanced security.

In [36]:
answer = response.choices[0].message.content

In [37]:
user_message = {
    "role": "user",
    "content": current_question
}

assistant_message = {
    "role": "assistant",
    "content": answer
}

print(user_message)
print(assistant_message)

{'role': 'user', 'content': 'What are its key features?'}
{'role': 'assistant', 'content': 'API connection successful.'}


## 9. Session / Conversation Management

### Definition

A session identifies a particular conversation.

For example:

session_id:

"abc123"

All messages belonging to this conversation use the same session
or conversation ID.

Example:

Conversation 1:
    session_id = abc123

Conversation 2:
    session_id = xyz789

This prevents messages from different conversations from being mixed.

In [38]:
session_id = "abc123"

print("Session ID:", session_id)

Session ID: abc123


## 10. Complete Chat Flow

The complete conversational RAG flow is:

User sends a message
        ↓
Identify session
        ↓
Load previous history
        ↓
Rewrite follow-up question
        ↓
Retrieve relevant documents
        ↓
Build prompt
        ↓
Generate answer
        ↓
Save user message
        ↓
Save assistant message
        ↓
Return answer

In [39]:
def chat(
    session_id,
    current_question,
    conversation_history,
    retrieved_documents
):
    history_text = ""

    for message in conversation_history:
        history_text += (
            f"{message['role'].capitalize()}: "
            f"{message['content']}\n"
        )

    context_text = "\n\n".join(retrieved_documents)

    prompt = f"""
You are a helpful document assistant.

Conversation history:
{history_text}

Retrieved document context:
{context_text}

Current question:
{current_question}

Answer using the retrieved context.
If the answer is not available, say you do not have enough information.
"""

    response = client_llm.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    conversation_history.append({
        "role": "user",
        "content": current_question
    })

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [40]:
answer = chat(
    session_id=session_id,
    current_question=current_question,
    conversation_history=conversation_history,
    retrieved_documents=retrieved_documents
)

print(answer)

**Key Features of the SmartHome Hub**

1. **Universal Compatibility** – Works with all major smart‑home brands.  
2. **AI‑Powered Assistant** – Learns user habits for proactive home management.  
3. **Enhanced Security** – Military‑grade encryption for data protection.
